# 03 – Between-Lines Detection

Run opponent line detection and receiver candidate identification.
Visualise the between-lines zone and example freeze frames.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_competition
from src.features.buildup import filter_buildup
from src.features.line_detection import detect_opponent_lines
from src.features.receiver import detect_receiver_candidates


In [ ]:
events, frames, lineups, matches = load_competition(competition_id=55, season_id=43)
buildup_df = filter_buildup(events)
line_df = detect_opponent_lines(frames, buildup_df)
print(f'Line data available for {len(line_df):,} events')
display(line_df.describe())


## Line gap distribution


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
line_df['defensive_line_x'].hist(bins=30, ax=axes[0])
axes[0].set_title('Defensive Line X')
line_df['midfield_line_x'].hist(bins=30, ax=axes[1])
axes[1].set_title('Midfield Line X')
line_df['line_gap_depth'].hist(bins=30, ax=axes[2])
axes[2].set_title('Line Gap Depth')
plt.tight_layout()
plt.show()


## Receiver candidates


In [ ]:
receivers_df = detect_receiver_candidates(frames, buildup_df, line_df)
print(f'Receiver candidates: {len(receivers_df):,}')
print(f'Between-lines candidates: {receivers_df["between_lines"].sum():,}')
print(f'Centrality breakdown:')
print(receivers_df[receivers_df['between_lines']]['centrality_zone'].value_counts())


## Example freeze frame (first event with between-lines candidate)


In [ ]:
from src.features.lane import compute_lane_features
from src.features.findability import compute_findability
from src.viz.plots import plot_freeze_frame

candidates_df = compute_lane_features(receivers_df, frames, buildup_df)
event_scores, candidates_scored = compute_findability(candidates_df)

# Pick first event with a findable option
findable_events = event_scores[event_scores['findable_option_available'] == 1]['event_id']
events_with_frames = set(frames['event_id'].unique())
sample_ids = [eid for eid in findable_events if eid in events_with_frames]

if sample_ids:
    eid = sample_ids[0]
    fig = plot_freeze_frame(
        event_id=eid,
        frames_df=frames,
        buildup_df=buildup_df,
        line_df=line_df,
        candidates_scored_df=candidates_scored,
    )
    plt.show()
else:
    print('No findable events with frame data in this sample.')
